# Cardiovascular Disease Classification — Model Training

**ML Assignment 2 | BITS Pilani M.Tech AIML/DSE**

Dataset: [Cardiovascular Disease dataset](https://www.kaggle.com/datasets/sulianova/cardiovascular-disease-dataset) (`cardio_train.csv`) — 70,000 patient records, target `cardio` (0 = no disease, 1 = disease).

Run all cells top to bottom. This notebook must be executed **on BITS Virtual Lab** (see assignment Section 1) and a screenshot of that execution submitted.

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef, confusion_matrix
)

RANDOM_STATE = 42


## 1. Load data

In [ ]:
df = pd.read_csv("cardio_train.csv", sep=";")
print("Raw shape:", df.shape)
df.head()


In [ ]:
print(df.isnull().sum())
print(df["cardio"].value_counts())
df.describe()


## 2. Clean invalid entries

This dataset is known to contain data-entry errors: negative/absurd blood pressure readings, `ap_hi < ap_lo`, and unrealistic height/weight. We drop physiologically impossible rows rather than impute them, since these look like typos, not missing data.

In [ ]:
before = len(df)
df = df[(df.ap_hi > 0) & (df.ap_hi <= 250)]
df = df[(df.ap_lo > 0) & (df.ap_lo <= 200)]
df = df[df.ap_hi >= df.ap_lo]
df = df[(df.height >= 120) & (df.height <= 220)]
df = df[(df.weight >= 30) & (df.weight <= 200)]
print(f"Dropped {before - len(df)} rows ({(before-len(df))/before:.1%})")
print("Clean shape:", df.shape)


## 3. Feature engineering

The raw dataset has only 11 usable features (excluding `id`/target), below the assignment's 12-feature minimum. We engineer three physiologically meaningful features instead of just padding the column count:

- `age_years` — raw `age` is in days, hard to interpret
- `bmi` — weight and height combined the way clinicians actually read them
- `pulse_pressure` — `ap_hi - ap_lo`, a known cardiovascular risk indicator on its own

In [ ]:
df["age_years"] = (df["age"] / 365.25).round(1)
df["bmi"] = df["weight"] / ((df["height"] / 100) ** 2)
df["pulse_pressure"] = df["ap_hi"] - df["ap_lo"]
df = df.drop(columns=["id", "age"])

feature_cols = [c for c in df.columns if c != "cardio"]
print(f"Feature count: {len(feature_cols)}")
print(feature_cols)


## 4. Train/test split + scaling

In [ ]:
X = df[feature_cols]
y = df["cardio"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # never fit_transform on test data


## 5. Train the 5 classification models

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(max_depth=8, random_state=RANDOM_STATE),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=15),
    "Naive Bayes": GaussianNB(),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=10, random_state=RANDOM_STATE),
}

trained_models = {}
for name, m in models.items():
    m.fit(X_train_scaled, y_train)
    trained_models[name] = m
    print(f"Trained: {name}")


## 6. Evaluate: Accuracy, AUC, Precision, Recall, F1, MCC

In [ ]:
results = []
cms = {}
for name, m in trained_models.items():
    y_pred = m.predict(X_test_scaled)
    y_proba = m.predict_proba(X_test_scaled)[:, 1]
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC": roc_auc_score(y_test, y_proba),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "MCC": matthews_corrcoef(y_test, y_pred),
    })
    cms[name] = confusion_matrix(y_test, y_pred)

results_df = pd.DataFrame(results).set_index("Model").round(4)
results_df


## 7. Confusion matrices

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()
for ax, (name, cm) in zip(axes, cms.items()):
    sns.heatmap(cm, annot=True, fmt="d", cmap="Purples", ax=ax,
                xticklabels=["No Disease", "Disease"],
                yticklabels=["No Disease", "Disease"])
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
axes[-1].axis("off")
plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150)
plt.show()


## 8. Save artifacts

Pickle every trained model plus the fitted scaler (needed so the Streamlit app preprocesses new uploads identically to training), and export a sample of the unscaled test set for the app's demo upload.

In [ ]:
os.makedirs("model", exist_ok=True)
with open("model/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)
with open("model/feature_cols.pkl", "wb") as f:
    pickle.dump(feature_cols, f)

filename_map = {
    "Logistic Regression": "logistic_regression",
    "Decision Tree": "decision_tree",
    "K-Nearest Neighbors": "knn",
    "Naive Bayes": "naive_bayes",
    "Random Forest": "random_forest",
}
for name, m in trained_models.items():
    with open(f"model/{filename_map[name]}.pkl", "wb") as f:
        pickle.dump(m, f)

test_df = X_test.copy()
test_df["cardio"] = y_test.values
test_sample = test_df.sample(n=min(2000, len(test_df)), random_state=RANDOM_STATE)
test_sample.to_csv("test_data.csv", index=False)

results_df.to_csv("model_comparison.csv")
print("All artifacts saved.")
